# Entrypoint

Este notebook cuida da leitura bruta, paralelismo de CPU e persistência na camada Silver:
1. importação dos dados
2. pré-processamento
3. extração das features necessárias

## 1. Importações e Configurações de Ambiente

In [26]:
import os
import ijson
import lizard
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from concurrent.futures import ProcessPoolExecutor
import multiprocessing

# Definindo caminhos relativos ao diretório pipeline
BRONZE_PATH = '../01_bronze/FormAI-v2.json'
SILVER_PATH = '../02_silver/formai_silver.parquet'

# Verificando a capacidade do seu i5-11300H
WORKERS = os.cpu_count()
print(f"Threads lógicas disponíveis para multiprocessamento: {WORKERS}")

Threads lógicas disponíveis para multiprocessamento: 8


## 2. Definição da Função Worker (Isolada para Multiprocessamento)

In [27]:
def process_code_record(record):
    """
    Função pura que roda isolada em cada núcleo do processador.
    Recebe um dicionário limpo, analisa a string C com Lizard e a descarta,
    retornando apenas os metadados e as features numéricas.
    """
    code_string = record.get('source_code', '')
    
    # Análise estática em memória (CPU Bound)
    analysis = lizard.analyze_file.analyze_source_code("temp.c", code_string)
    
    func_count = len(analysis.function_list)
    if func_count > 0:
        total_params = sum(len(func.parameters) for func in analysis.function_list)
        avg_params = round(total_params / func_count, 2)
    else:
        avg_params = 0.0
        
    # Retorna o dicionário processado (o texto pesado foi liberado da memória)
    return {
        'file_name': record.get('file_name', 'N/A'),
        'category': record.get('category', 'N/A'),
        'error_type': record.get('error_type', 'N/A'),
        'num_lines': float(record.get('num_lines', 0)),
        'cyclomatic_complexity': float(record.get('cyclomatic_complexity', 0.0)),
        'token_count': float(analysis.token_count),
        'function_count': float(func_count),
        'avg_parameter_count': float(avg_params)
    }

## 3. Orquestrador de Lotes e Escritor Parquet (Pipeline Principal)

In [29]:
def build_silver_layer(json_path, parquet_path, batch_size=15000):
    """
    Orquestrador de Lotes e Escritor Parquet
    """
    print("Iniciando a extração massiva paralelizada para a camada Silver...")
    
    # Definimos um Schema estrito para o PyArrow otimizar o I/O do disco
    schema = pa.schema([
        ('file_name', pa.string()),
        ('category', pa.string()),
        ('error_type', pa.string()),
        ('num_lines', pa.float64()),
        ('cyclomatic_complexity', pa.float64()),
        ('token_count', pa.float64()),
        ('function_count', pa.float64()),
        ('avg_parameter_count', pa.float64())
    ])
    
    parquet_writer = None
    processed_total = 0
    
    with open(json_path, 'r', encoding='utf-8') as f:
        # ijson evita carregar 2.5GB de uma vez na RAM
        objects = ijson.items(f, 'item')
        batch: list[dict] = []
        
        # Inicia o Pool de processamento usando todas as threads do seu Intel Core i5
        mp_ctx = multiprocessing.get_context("fork")
        with ProcessPoolExecutor(max_workers=WORKERS, mp_context=mp_ctx) as executor:
            for obj in objects:
                # Criamos um dict mínimo para trafegar entre os processos (economiza serialização)
                minimal_record = {
                    'file_name': obj.get('file_name'),
                    'category': obj.get('category'),
                    'error_type': obj.get('error_type'),
                    'num_lines': obj.get('num_lines'),
                    'cyclomatic_complexity': obj.get('cyclomatic_complexity'),
                    'source_code': obj.get('source_code', '')
                }
                batch.append(minimal_record)
                
                # Quando o lote atinge N, despachamos para a CPU e gravamos no disco
                if len(batch) >= batch_size:
                    # O map divide os N itens nos 8 núcleos simultaneamente
                    results = list(executor.map(process_code_record, batch))
                    
                    # Converte para Pandas e depois PyArrow Table
                    df_batch = pd.DataFrame(results)
                    table = pa.Table.from_pandas(df_batch, schema=schema)
                    
                    # Se for o primeiro lote, cria o arquivo. Senão, faz append.
                    if parquet_writer is None:
                        parquet_writer = pq.ParquetWriter(parquet_path, schema)
                    parquet_writer.write_table(table)
                    
                    processed_total += len(batch)
                    print(f"Progresso: {processed_total} arquivos C processados e compactados.")
                    
                    # Limpa a RAM instantaneamente para o próximo lote
                    batch.clear()
            
            # Processa as sobras do loop final
            if batch:
                results = list(executor.map(process_code_record, batch))
                df_batch = pd.DataFrame(results)
                table = pa.Table.from_pandas(df_batch, schema=schema)
                
                if parquet_writer is None:
                    parquet_writer = pq.ParquetWriter(parquet_path, schema)
                parquet_writer.write_table(table)
                processed_total += len(batch)
                print(f"Progresso: {processed_total} arquivos C processados e compactados.")
                
    if parquet_writer:
        parquet_writer.close()
        
    print(f"\nCamada Silver finalizada com sucesso! Todos os dados salvos em: {parquet_path}")

#### Gatilho de execução:

In [30]:
N = 15_000
build_silver_layer(BRONZE_PATH, SILVER_PATH, batch_size=N)

Iniciando a extração massiva paralelizada para a camada Silver...
Progresso: 15000 arquivos C processados e compactados.
Progresso: 30000 arquivos C processados e compactados.
Progresso: 45000 arquivos C processados e compactados.
Progresso: 60000 arquivos C processados e compactados.
Progresso: 75000 arquivos C processados e compactados.
Progresso: 90000 arquivos C processados e compactados.
Progresso: 105000 arquivos C processados e compactados.
Progresso: 120000 arquivos C processados e compactados.
Progresso: 135000 arquivos C processados e compactados.
Progresso: 150000 arquivos C processados e compactados.
Progresso: 165000 arquivos C processados e compactados.
Progresso: 180000 arquivos C processados e compactados.
Progresso: 195000 arquivos C processados e compactados.
Progresso: 210000 arquivos C processados e compactados.
Progresso: 225000 arquivos C processados e compactados.
Progresso: 240000 arquivos C processados e compactados.
Progresso: 255000 arquivos C processados e c